In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
print(ROOT_DIR)

c:\Users\kuchbhe\Desktop\workspace_1\travelara-cd-v2


In [2]:
from __future__ import annotations
import pandas as pd
import math
from datetime import datetime, timedelta
from app.schemas import (
    PlanningRequest, POI, StructuredIntent, Itinerary, DayPlan, ItineraryStop, ItineraryScore
)
from app.clustering.cluster import (
    haversine_km, cluster_pois, group_by_cluster, score_all_pois
)
from app.config import settings
from app.utils.save import save_artifact
from app.providers.provider import run_retrieval

from app.clustering.filter import *

In [3]:
from app.schemas import (
    StructuredIntent,
    Preferences,
    Constraints,
)

intent = StructuredIntent(
    destination="Tokyo",
    days=5,
    stay_location="Shinjuku",
    is_international=True,
    budget="medium",
    preferences=Preferences(
        museums=0.8,
        food=0.8,
        nightlife=0.0,
        nature=0.0,
        shopping=0.0,
        arts=0.0,
        history=0.0,
        wellness=0.0,
    ),
    constraints=Constraints(
        walking_limit_km=5.5,
        must_visit=[],
        avoid=[],
        budget_per_day_usd=None,
    ),
)

In [4]:
import json
# with open(r'C:\Users\kuchbhe\Desktop\workspace_1\travelara-cd-v2\experiments\runs\123456\POIS.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)

# pois = [POI.model_validate(item) for item in data]

pois_GA, lat, lon = await run_retrieval(
    source="GA",
    intent=intent,
    debug=True
)

pois_FS, lat, lon = await run_retrieval(
    source="FS",
    intent=intent,
    debug=True
)
pois = pois_GA + pois_FS
save_artifact('123456', 'POIS', pois)
sorted_pois = score_all_pois(pois, intent)


=== RETRIEVAL START ===
Provider: GA
Destination: Tokyo
Coordinates: (35.6895, 139.69171)
Preferences: {'museums': 0.8, 'food': 0.8, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 0.0, 'wellness': 0.0}
Processed: museums
Processed: food

=== DEDUPLICATION ===
Input POIs: 847
Output POIs: 755
Dropped: 92
By Category: {'museums': 3, 'food': 89}

=== AFTER DEDUP ===
Total POIs: 755
{'museums': 352, 'food': 403}

=== MUST VISIT FILTER ===
Must Visit Targets: []
Matched POIs: 0

=== AVOID FILTER ===
Avoid Categories: set()
Removed: 0

=== FINAL RESULT ===
Total POIs: 755
Must Visit POIs: 0
Regular POIs: 755
{'museums': 352, 'food': 403}


=== RETRIEVAL START ===
Provider: FS
Destination: Tokyo
Coordinates: (35.6895, 139.69171)
Preferences: {'museums': 0.8, 'food': 0.8, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 0.0, 'wellness': 0.0}
Processed: museums
Processed: food

=== DEDUPLICATION ===
Input POIs: 98
Output POIs: 49
Dropped: 49


In [ ]:
cluster_map = cluster_pois(sorted_pois)

In [6]:
from collections import defaultdict

def compute_cluster_scores(
    pois: list[POI],
    cluster_map: dict[str, int]
) -> dict[int, float]:

    cluster_scores = defaultdict(float)

    for poi in pois:
        cluster_id = cluster_map[poi.id]

        score = (
            poi.utility_score.raw_score
            if poi.utility_score is not None
            else 0.0
        )

        cluster_scores[cluster_id] += score

    scored_cluster = dict(cluster_scores)
    return sorted(scored_cluster.items(),
                 key= lambda x: x[1],
                 reverse=True)

compute_cluster_scores(sorted_pois, cluster_map)

[(28, 240.83515983809068),
 (23, 171.42169573931045),
 (32, 125.86047142747243),
 (16, 120.36822179963315),
 (53, 119.88851515454088),
 (26, 117.69937626412778),
 (18, 73.11044184760424),
 (19, 63.88543424606455),
 (25, 63.51822905876647),
 (15, 63.462771886794926),
 (9, 61.432617175242356),
 (24, 60.842711877937035),
 (10, 58.05950433853459),
 (22, 56.95823744863283),
 (44, 54.446493551898726),
 (3, 51.78366319421778),
 (13, 48.23296695479073),
 (38, 47.029645634844755),
 (43, 46.936449192204286),
 (41, 46.51917185511304),
 (31, 43.933392075729095),
 (20, 42.396456565289334),
 (39, 40.977321869434554),
 (14, 40.34949267599161),
 (34, 40.183210229202565),
 (57, 40.04443368533256),
 (61, 39.425719675339316),
 (27, 39.40792376299456),
 (6, 38.24772981596069),
 (2, 38.12673819634577),
 (7, 38.07026903601385),
 (17, 37.03125032152567),
 (52, 36.92779466481639),
 (36, 36.719433828954486),
 (54, 36.2875070148846),
 (0, 35.519585005911644),
 (29, 34.2484482023933),
 (33, 34.162772511266596),


In [ ]:
from pydantic import BaseModel
class ClusterMetrics(BaseModel):
    cluster_id: int

    sum_score: float
    max_score: float
    p90_score: float

    size: int
    density: float

    survival_score: float
    protected: bool


from collections import defaultdict
import numpy as np


def _normalize(values: dict[int, float]) -> dict[int, float]:
    if not values:
        return {}

    vmin = min(values.values())
    vmax = max(values.values())

    if vmax == vmin:
        return {k: 1.0 for k in values}

    return {
        k: (v - vmin) / (vmax - vmin)
        for k, v in values.items()
    }


def compute_cluster_scores(
    pois: list[POI],
    cluster_map: dict[str, int],
    protected_top_n: int = 50,
):
    # cluster_id -> pois
    clusters = defaultdict(list)

    for poi in pois:
        clusters[cluster_map[poi.id]].append(poi)

    # top global POIs
    sorted_pois = sorted(
        pois,
        key=lambda p: (
            p.utility_score.raw_score
            if p.utility_score is not None
            else 0.0
        ),
        reverse=True,
    )

    protected_pois = {
        p.id
        for p in sorted_pois[:protected_top_n]
    }

    cluster_stats = {}

    for cluster_id, members in clusters.items():

        scores = [
            p.utility_score.raw_score
            for p in members
            if p.utility_score is not None
        ]

        if not scores:
            scores = [0.0]

        cluster_stats[cluster_id] = {
            "cluster_id": cluster_id,
            "sum_score": float(np.sum(scores)),
            "max_score": float(np.max(scores)),
            "p90_score": float(np.percentile(scores, 90)),
            "size": len(members),

            # temporary density proxy
            "density": (
                float(np.sum(scores))
                / max(len(members), 1)
            ),

            "protected": any(
                p.id in protected_pois
                for p in members
            ),
        }

    # normalize metrics
    norm_sum = _normalize({
        cid: c["sum_score"]
        for cid, c in cluster_stats.items()
    })

    norm_max = _normalize({
        cid: c["max_score"]
        for cid, c in cluster_stats.items()
    })

    norm_p90 = _normalize({
        cid: c["p90_score"]
        for cid, c in cluster_stats.items()
    })

    norm_density = _normalize({
        cid: c["density"]
        for cid, c in cluster_stats.items()
    })

    # survival score
    for cid, c in cluster_stats.items():

        c["survival_score"] = (
            0.40 * norm_sum[cid]
            + 0.25 * norm_max[cid]
            + 0.25 * norm_p90[cid]
            + 0.10 * norm_density[cid]
        )

    ranked_clusters = sorted(
        cluster_stats.values(),
        key=lambda c: c["survival_score"],
        reverse=True,
    )

    return ranked_clusters



In [8]:
ranked_clusters = compute_cluster_scores(
    pois,
    cluster_map
)

survival_scores = [
    c["survival_score"]
    for c in ranked_clusters
]

threshold = np.percentile(
    survival_scores,
    60
)

selected_clusters = [
    c
    for c in ranked_clusters
    if (
        c["protected"]
        or c["survival_score"] >= threshold
    )
]

In [ ]:
selected_cluster_ids = {
    c["cluster_id"]
    for c in selected_clusters
}

selected_pois = [
    poi
    for poi in pois
    if (
        cluster_map[poi.id] in selected_cluster_ids
        and poi.wiki_and_media
        and poi.wiki_and_media.get("wikidata")
    )
]

In [49]:
import asyncio
from urllib.parse import quote

import httpx

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

HEADERS = {
    "User-Agent": "Travelara/0.1 (your@email.com)",
    "Accept": "application/json",
}


async def enrich_selected_pois(selected_pois: list[POI]):

    qid_to_pois = {}

    for poi in selected_pois:
        qid = (poi.wiki_and_media or {}).get("wikidata")

        if qid:
            qid_to_pois.setdefault(qid, []).append(poi)

    if not qid_to_pois:
        return

    async with httpx.AsyncClient(
        headers=HEADERS,
        timeout=30,
        follow_redirects=True,
    ) as client:

        qids = list(qid_to_pois.keys())

        #
        # STEP 1
        # Batch fetch Wikidata
        #

        qid_to_page = {}
        qid_to_description = {}
        qid_to_label = {}

        for i in range(0, len(qids), 50):

            batch = qids[i:i + 50]

            r = await client.get(
                WIKIDATA_API,
                params={
                    "action": "wbgetentities",
                    "format": "json",
                    "ids": "|".join(batch),
                    "props": "labels|descriptions|sitelinks",
                    "languages": "en",
                    "maxlag": 5,
                },
            )

            r.raise_for_status()

            entities = r.json()["entities"]

            for qid, entity in entities.items():

                #
                # Label
                #

                label = (
                    entity.get("labels", {})
                    .get("en", {})
                    .get("value")
                )

                if label:
                    qid_to_label[qid] = label

                #
                # Description
                #

                description = (
                    entity.get("descriptions", {})
                    .get("en", {})
                    .get("value")
                )

                if description:
                    qid_to_description[qid] = description

                #
                # Wikipedia page
                #

                sitelinks = entity.get("sitelinks", {})

                title = None
                lang = None

                # Prefer English
                if "enwiki" in sitelinks:
                    lang = "en"
                    title = sitelinks["enwiki"]["title"]

                # Otherwise use first available Wikipedia
                else:
                    for site, info in sitelinks.items():
                        if site.endswith("wiki"):
                            lang = site[:-4]  # jawiki -> ja
                            title = info["title"]
                            break

                if title:
                    qid_to_page[qid] = (lang, title)

        #
        # STEP 2
        # Fetch Wikipedia summaries
        #

        sem = asyncio.Semaphore(5)

        async def fetch_summary(qid: str, lang: str, title: str):

            async with sem:

                url = (
                    f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/"
                    + quote(title)
                )

                r = await client.get(url)

                if r.status_code != 200:
                    return qid, None, None

                data = r.json()

                image_url = (
                    data.get("thumbnail", {}).get("source")
                    or data.get("originalimage", {}).get("source")
                )

                summary = (
                    data.get("extract")
                    or data.get("description")
                    or data.get("title")
                )

                return qid, summary, image_url

        tasks = [
            fetch_summary(qid, lang, title)
            for qid, (lang, title) in qid_to_page.items()
        ]

        summaries = await asyncio.gather(*tasks)

        summary_map = {
            qid: (summary, img)
            for qid, summary, img in summaries
            if summary
        }

        #
        # STEP 3
        # Populate POIs
        #

        for qid, pois in qid_to_pois.items():

            summary, img = summary_map.get(qid, (None, None))

            #
            # Fallback 1: Wikidata description
            #

            if not summary:
                summary = qid_to_description.get(qid)

            #
            # Fallback 2: Deterministic description from POI metadata
            #

            if not summary:

                poi = pois[0]

                label = qid_to_label.get(qid) or poi.name

                category = getattr(poi, "category", None)

                city = None
                country = None

                if getattr(poi, "address", None):
                    city = getattr(poi.address, "city", None)
                    country = getattr(poi.address, "country", None)

                parts = [label]

                if category:
                    parts.append(f"is a {category.lower()}")

                location = ", ".join(
                    x for x in [city, country] if x
                )

                if location:
                    parts.append(f"located in {location}")

                summary = " ".join(parts)

            #
            # Final fallback
            #

            if not summary:
                summary = pois[0].name

            for poi in pois:
                poi.wiki_enrichment = {
                    "description": summary,
                    "img_url": img,
                }

In [50]:
await enrich_selected_pois(selected_pois)

for poi in selected_pois:
    print(poi.name)
    print(poi.wiki_enrichment)
    print("-" * 80)

SOMPO美術館
{'description': "The Sompo Museum of Art  is an art museum in Shinjuku, Tokyo, Japan. It is owned by the Japanese insurance company SOMPO and is located next to the company's headquarters. It started as the Seiji Togo Memorial Sompo Japan Nipponkoa Museum of Art in 1976 and gradually expanded. The current six-storey building was completed in 2020.", 'img_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Sompo_Museum_of_Art_2024-01-25.jpg/330px-Sompo_Museum_of_Art_2024-01-25.jpg'}
--------------------------------------------------------------------------------
明治神宮宝物殿
{'description': 'museum in Shibuya, Tokyo, Japan', 'img_url': None}
--------------------------------------------------------------------------------
明治神宮ミュージアム
{'description': 'museum in Tokyo, Japan', 'img_url': None}
--------------------------------------------------------------------------------
佐藤美術館
{'description': 'museum in Japan', 'img_url': None}
--------------------------------------------